# Tutorial 4: Charged Species

**Calculating ions: cations, anions, and protonation**

## 🎓 Learning Objectives

By the end of this tutorial, you will:
- ✅ Set up molecules with non-zero charge (and the right multiplicity)
- ✅ Compare neutral, cation, and anion calculations in the NH₃ family
- ✅ Compute an ionization energy, a proton affinity, a deprotonation energy, and an electron affinity
- ✅ Apply charge/multiplicity rules correctly (including radical ions)
- ✅ See why anions need diffuse basis functions — and electron correlation

## ⏱️ Estimated Time
**20-25 minutes** (every calculation here takes only seconds)

## 📚 Prerequisites
- Tutorial 1 completed
- Tutorial 3 (multiplicity, RHF vs. UHF) recommended
- Understanding of ions and electron counting
- Familiarity with Lewis structures

---

**Let's explore charged molecules!** ⚡

---

## Introduction: Why Calculate Charged Species?

### Real-World Importance
- **Biology:** Protonation states at different pH
- **Catalysis:** Charged intermediates in reactions
- **Mass spectrometry:** Ionized molecules
- **Electrochemistry:** Oxidation and reduction

### The Challenge
Charged species have:
- A different number of electrons than the neutral molecule
- Possibly a different multiplicity (removing one electron from a closed shell gives a radical)
- Often a different geometry
- Anions need basis sets with **diffuse functions** — their extra electron is loosely held

### Today's Example: Ammonia (NH₃) and Its Relatives
**A closed-shell, isoelectronic series (10 electrons each):**
1. **NH₃** — neutral ammonia
2. **NH₄⁺** — ammonium cation (NH₃ + H⁺)
3. **NH₂⁻** — amide anion (NH₃ − H⁺)

**Two open-shell radicals (9 electrons each):**

4. **NH₃•⁺** — ammonia radical cation (NH₃ minus one electron) → ionization energy
5. **NH₂•** — amino radical (NH₂⁻ minus one electron) → electron affinity

All calculations run **in your own Python session** (this notebook's kernel) and each one finishes in seconds.

---

## Step 1: Setup

Import QuantUI and define two unit conversions we will use throughout.

### Run the cell below ⬇️

In [ ]:
import contextlib
import importlib.util
import io

import numpy as np

from quantui import Molecule, config, parse_xyz_input, run_in_session

HARTREE_TO_EV = 27.211386  # 1 Hartree in electron-volts
HARTREE_TO_KCAL = 627.5095  # 1 Hartree in kcal/mol

if importlib.util.find_spec("pyscf") is None:
    print("⚠️  PySCF is not installed — the calculation cells below will fail.")
    print("   PySCF runs on Linux, macOS, or WSL (not native Windows).")
else:
    print("✓ QuantUI and PySCF are ready")
    print("✓ Calculations will run in this notebook's own Python kernel")

print(f"\nBasis sets QuantUI supports: {', '.join(config.SUPPORTED_BASIS_SETS)}")

---

## Step 2: Understanding Charge and Electrons

### Electron Count Formula
$$N_\text{electrons} = \sum_i Z_i - Q$$

Where:
- $Z_i$ = atomic numbers
- $Q$ = molecular charge

### Examples
| Species | Atoms | Nuclear charge | Molecular charge | Electrons |
|---------|-------|----------------|------------------|-----------|
| NH₃  | N(7) + 3H(1) | 10 | 0  | 10 |
| NH₄⁺ | N(7) + 4H(1) | 11 | +1 | 10 |
| NH₂⁻ | N(7) + 2H(1) | 9  | −1 | 10 |
| NH₃•⁺ | N(7) + 3H(1) | 10 | +1 | 9 |
| NH₂• | N(7) + 2H(1) | 9  | 0  | 9 |

**NH₃, NH₄⁺ and NH₂⁻ all have 10 electrons (isoelectronic) but different structures!**

### Run the cell below ⬇️

In [ ]:
# Count electrons for each species by hand
species_info = {
    "NH3": {"atoms": {"N": 1, "H": 3}, "charge": 0},
    "NH4+": {"atoms": {"N": 1, "H": 4}, "charge": +1},
    "NH2-": {"atoms": {"N": 1, "H": 2}, "charge": -1},
    "NH3•+": {"atoms": {"N": 1, "H": 3}, "charge": +1},
    "NH2•": {"atoms": {"N": 1, "H": 2}, "charge": 0},
}
atomic_numbers = {"N": 7, "H": 1}

print("ELECTRON COUNTING")
print("=" * 60)
print(f"{'Species':8s} {'Nuclear charge':>15s} {'Charge':>8s} {'Electrons':>10s}")
for name, info in species_info.items():
    nuclear_charge = sum(atomic_numbers[el] * n for el, n in info["atoms"].items())
    electrons = nuclear_charge - info["charge"]
    print(f"{name:8s} {nuclear_charge:15d} {info['charge']:8d} {electrons:10d}")
print("=" * 60)
print("💡 NH3, NH4+ and NH2- are isoelectronic (10 electrons).")
print("💡 The two radicals have 9 electrons — an odd number.")

### ✅ Checkpoint 1

**Key concepts:**
- Charge changes the electron count
- Positive charge → fewer electrons
- Negative charge → more electrons
- Isoelectronic species have the same electron count
- Adding or removing a **proton** (H⁺) changes the charge but *not* the electron count; adding or removing an **electron** changes both

---

## Step 3: Define Geometries

Each species gets its own structure. These are fixed, reasonable (experiment-like) geometries — **not optimized** at our level of theory — so the energies below are *single-point* energies.

### NH₃ (Trigonal pyramidal)
- One lone pair on N
- N–H = 1.012 Å, H–N–H ≈ 106.7°

### NH₄⁺ (Tetrahedral)
- No lone pair
- N–H = 1.030 Å, H–N–H = 109.5°

### NH₂⁻ (Bent)
- Two lone pairs on N
- N–H = 1.030 Å, H–N–H = 104°

The radicals reuse these structures: NH₃•⁺ uses the NH₃ geometry and NH₂• uses the NH₂⁻ geometry. Removing or adding an electron **without letting the nuclei move** is called a **vertical** process.

### Run the cell below ⬇️

In [ ]:
# Cartesian coordinates in Angstrom (element  x  y  z)

# NH3 - pyramidal
nh3_xyz = '''N   0.0000   0.0000   0.0677
H   0.0000   0.9375  -0.3134
H  -0.8119  -0.4688  -0.3134
H   0.8119  -0.4688  -0.3134'''

# NH4+ - tetrahedral
nh4_xyz = '''N   0.0000   0.0000   0.0000
H   0.5947   0.5947   0.5947
H   0.5947  -0.5947  -0.5947
H  -0.5947   0.5947  -0.5947
H  -0.5947  -0.5947   0.5947'''

# NH2- - bent
nh2_xyz = '''N   0.0000   0.0000   0.0798
H   0.0000   0.8117  -0.5543
H   0.0000  -0.8117  -0.5543'''

atoms_nh3, coords_nh3 = parse_xyz_input(nh3_xyz)
atoms_nh4, coords_nh4 = parse_xyz_input(nh4_xyz)
atoms_nh2, coords_nh2 = parse_xyz_input(nh2_xyz)

# Check one bond length per species
for label, coords in [("NH3", coords_nh3), ("NH4+", coords_nh4), ("NH2-", coords_nh2)]:
    r_nh = np.linalg.norm(np.array(coords[1]) - np.array(coords[0]))
    print(f"{label:5s}: {len(coords)} atoms, N-H = {r_nh:.3f} Å")

### Optional: Look at the structures in 3D

This needs a 3D viewer (py3Dmol). If it is not installed the cell just prints a note — nothing else in the tutorial depends on it.

In [ ]:
try:
    from quantui import display_molecule

    for atoms, coords, charge in [
        (atoms_nh3, coords_nh3, 0),
        (atoms_nh4, coords_nh4, +1),
        (atoms_nh2, coords_nh2, -1),
    ]:
        display_molecule(Molecule(atoms=atoms, coordinates=coords, charge=charge), width=300, height=250)
except ImportError:
    print("3D viewer not available — skipping (this is optional).")

---

## Step 4: Determine Multiplicity for Each Species

**Rule:** multiplicity = 2S + 1. An even electron count needs an odd multiplicity (1, 3, …); an odd count needs an even one (2, 4, …).

| Species | Electrons | Unpaired | Multiplicity | SCF method |
|---------|-----------|----------|--------------|------------|
| NH₃   | 10 | 0 | 1 (singlet) | RHF |
| NH₄⁺  | 10 | 0 | 1 (singlet) | RHF |
| NH₂⁻  | 10 | 0 | 1 (singlet) | RHF |
| NH₃•⁺ | 9  | 1 | 2 (doublet) | UHF |
| NH₂•  | 9  | 1 | 2 (doublet) | UHF |

**Closed-shell singlets → RHF. Radicals (doublets) → UHF** (see Tutorial 3).

### Run the cell below ⬇️

In [ ]:
# Create molecule objects with the correct charge and multiplicity
mol_nh3 = Molecule(atoms=atoms_nh3, coordinates=coords_nh3, charge=0, multiplicity=1)
mol_nh4 = Molecule(atoms=atoms_nh4, coordinates=coords_nh4, charge=+1, multiplicity=1)
mol_nh2 = Molecule(atoms=atoms_nh2, coordinates=coords_nh2, charge=-1, multiplicity=1)
mol_nh3_cation = Molecule(atoms=atoms_nh3, coordinates=coords_nh3, charge=+1, multiplicity=2)
mol_nh2_radical = Molecule(atoms=atoms_nh2, coordinates=coords_nh2, charge=0, multiplicity=2)

molecules = {
    "NH3": mol_nh3,
    "NH4+": mol_nh4,
    "NH2-": mol_nh2,
    "NH3•+": mol_nh3_cation,
    "NH2•": mol_nh2_radical,
}

print("MOLECULE SUMMARY")
print("=" * 60)
print(f"{'Species':8s} {'Formula':8s} {'Electrons':>10s} {'Charge':>7s} {'Mult.':>6s}")
for name, mol in molecules.items():
    print(
        f"{name:8s} {mol.get_formula():8s} {mol.get_electron_count():10d} "
        f"{mol.charge:7d} {mol.multiplicity:6d}"
    )
print("=" * 60)
print("✓ All molecules validated")

QuantUI checks that charge and multiplicity are consistent. Watch what happens if we forget that NH₃•⁺ is a radical and ask for a singlet:

In [ ]:
try:
    Molecule(atoms=atoms_nh3, coordinates=coords_nh3, charge=+1, multiplicity=1)
except ValueError as err:
    print("QuantUI rejected the input:\n")
    print(err)

### ✅ Checkpoint 2

**Verify:**
- NH₃, NH₄⁺, NH₂⁻: 10 electrons, multiplicity 1
- NH₃•⁺, NH₂•: 9 electrons, multiplicity 2
- Charges: 0, +1, −1, +1, 0
- A singlet NH₃•⁺ is impossible (9 electrons cannot all be paired)

---

## Step 5: Run the Calculations

Each call to `run_in_session(molecule, method=..., basis=...)` runs a PySCF calculation **right here in your Python session** and returns a `SessionResult` with the energy, convergence information, orbital energies, Mulliken charges, and more.

We start with the **6-31G** basis: RHF for the closed-shell species and UHF for the two radicals.

### Note on Anions
- 6-31G has **no diffuse functions**, so the anion NH₂⁻ is described poorly.
- We use it first on purpose, then repeat with **6-31+G\*** in Step 8 to see the difference.

### Run the cell below ⬇️ (a few seconds)

In [ ]:
def run_calc(mol, method, basis):
    # Run one single-point calculation in this kernel.
    # verbose=0 and a log buffer keep PySCF's detailed output out of the notebook;
    # warm_start=False makes every SCF start from scratch (reproducible results).
    log = io.StringIO()
    with contextlib.redirect_stdout(log):
        return run_in_session(
            mol, method=method, basis=basis, verbose=0, progress_stream=log, warm_start=False
        )


def scf_method(mol):
    # RHF for closed-shell singlets, UHF for open-shell species
    return "RHF" if mol.multiplicity == 1 else "UHF"


basis = "6-31G"
results = {}
for name, mol in molecules.items():
    results[name] = run_calc(mol, scf_method(mol), basis)

print(f"RESULTS ({basis})")
print("=" * 72)
print(f"{'Species':8s} {'Method':6s} {'Energy (Ha)':>14s} {'Conv.':>6s} {'Iter.':>6s} {'<S^2>':>7s}")
for name, res in results.items():
    s2 = f"{res.spin_square:.3f}" if res.spin_square is not None else "—"
    print(
        f"{name:8s} {res.method:6s} {res.energy_hartree:14.6f} "
        f"{str(res.converged):>6s} {res.n_iterations:6d} {s2:>7s}"
    )
print("=" * 72)

### ✅ Checkpoint 3

**Expected (RHF or UHF / 6-31G, geometries above):**

| Species | Energy (Ha) | ⟨S²⟩ |
|---------|-------------|------|
| NH₃   | −56.1611 | — |
| NH₄⁺  | −56.5154 | — |
| NH₂⁻  | −55.4556 | — |
| NH₃•⁺ | −55.8248 | 0.757 |
| NH₂•  | −55.5323 | 0.757 |

- Every calculation should report `converged = True`.
- For a doublet the exact ⟨S²⟩ is S(S+1) = 0.75. UHF gives 0.757 — very little spin contamination, so the UHF results are trustworthy.

---

## Step 6: Can We Rank the Total Energies?

Look at the energies: NH₄⁺ is the most negative and NH₂⁻ the least negative. **That does *not* mean NH₄⁺ is the "most stable" species.**

A total energy measures how much energy it would take to pull the molecule apart into bare nuclei and free electrons. NH₄⁺ has an extra proton that attracts the 10 electrons, so its total energy is naturally lower. Total energies are only comparable for systems with **the same atoms and the same number of electrons** (e.g. two conformers of one molecule).

To make chemically meaningful comparisons we must write a **balanced reaction** — that is Step 7.

First, let's see where the charge sits. **Mulliken charges** split the electron density among the atoms (they are an approximate bookkeeping tool and change with basis set — we will see this in Step 8).

### Run the cell below ⬇️

In [ ]:
print(f"MULLIKEN CHARGES AND HOMO-LUMO GAPS ({basis})")
print("=" * 72)
print(f"{'Species':8s} {'q(N)':>7s} {'q(each H)':>10s} {'Sum':>6s} {'Gap (eV)':>9s}  Note")
for name, res in results.items():
    q = res.mulliken_charges
    q_n = q[0]  # nitrogen is atom 0 in every structure
    q_h = q[1]  # all H atoms are equivalent in these symmetric structures
    gap = f"{res.homo_lumo_gap_ev:.2f}" if res.homo_lumo_gap_ev is not None else "—"
    note = "alpha-spin gap (UHF)" if res.multiplicity and res.multiplicity > 1 else ""
    total = round(sum(q), 2) + 0.0  # + 0.0 avoids printing "-0.00"
    print(f"{name:8s} {q_n:7.2f} {q_h:10.2f} {total:6.2f} {gap:>9s}  {note}")
print("=" * 72)
print("💡 The Mulliken charges always add up to the total molecular charge.")

### What to Notice (6-31G values)

- **NH₄⁺:** q(N) ≈ −0.93 and each H ≈ +0.48. The "+1" of ammonium is spread over the **hydrogens**, not located on nitrogen — the formal charge on N in a Lewis structure is not a physical charge.
- **NH₂⁻:** q(N) ≈ −1.14. The extra negative charge piles up on nitrogen (each H is only ≈ +0.07).
- **NH₃•⁺:** removing an electron from NH₃ makes q(N) much less negative (−0.91 → −0.42): the electron came from the nitrogen lone pair.
- **HOMO-LUMO gaps:** NH₄⁺ (≈ 25.3 eV) > NH₃ (≈ 17.2 eV) > NH₂⁻ (≈ 14.1 eV). Extra positive charge binds the electrons more tightly; the anion's electrons are the most loosely held.

---

## Step 7: Ionization, Protonation and Deprotonation Energies

Now compare species through **balanced reactions**.

### Definitions (all at fixed geometries, electronic energies only)
- **Vertical ionization energy (IE):** NH₃ → NH₃•⁺ + e⁻
  $$IE = E(\text{NH}_3^{\bullet +}) - E(\text{NH}_3)$$
  Computing it as a difference of two SCF energies is called the **ΔSCF** method.
- **Koopmans' theorem** estimate: $IE \approx -\varepsilon_\text{HOMO}$ (the HOMO orbital energy of NH₃), with no calculation on the cation at all.
- **Proton affinity (PA):** NH₃ + H⁺ → NH₄⁺
  $$PA = E(\text{NH}_3) + E(\text{H}^+) - E(\text{NH}_4^+)$$
- **Deprotonation energy (gas-phase acidity):** NH₃ → NH₂⁻ + H⁺
  $$\Delta E_\text{deprot} = E(\text{NH}_2^-) + E(\text{H}^+) - E(\text{NH}_3)$$

💡 A bare proton has **no electrons**, so its electronic energy is exactly zero: $E(\text{H}^+) = 0$. That is why these reactions only need our molecular energies.

### Run the cell below ⬇️

In [ ]:
def homo_energy_ev(res):
    # Highest occupied orbital energy (eV) of a closed-shell (RHF) result
    mo_energy = np.asarray(res.mo_energy_hartree)
    mo_occ = np.asarray(res.mo_occ)
    return float(mo_energy[mo_occ > 0].max()) * HARTREE_TO_EV


def reaction_energies(res):
    # Energy differences (from a dict of SessionResults) used in this tutorial
    E = {name: r.energy_hartree for name, r in res.items()}
    return {
        "IE_dSCF_eV": (E["NH3•+"] - E["NH3"]) * HARTREE_TO_EV,
        "IE_Koopmans_eV": -homo_energy_ev(res["NH3"]),
        "PA_kcal": (E["NH3"] - E["NH4+"]) * HARTREE_TO_KCAL,
        "deprot_kcal": (E["NH2-"] - E["NH3"]) * HARTREE_TO_KCAL,
    }


rxn_631g = reaction_energies(results)

print(f"🔬 ENERGY ANALYSIS ({basis})")
print("=" * 60)
print("1️⃣  Vertical ionization energy of NH3")
print(f"    ΔSCF:      {rxn_631g['IE_dSCF_eV']:7.2f} eV")
print(f"    Koopmans:  {rxn_631g['IE_Koopmans_eV']:7.2f} eV")
print("2️⃣  Proton affinity of NH3 (NH3 + H+ → NH4+)")
print(f"    PA = {rxn_631g['PA_kcal']:.1f} kcal/mol  (energy released)")
print("3️⃣  Deprotonation energy of NH3 (NH3 → NH2- + H+)")
print(f"    ΔE = {rxn_631g['deprot_kcal']:+.1f} kcal/mol  (energy required)")
print("=" * 60)

### ✅ Checkpoint 4

**Expected (6-31G):**

| Quantity | This calculation | Experiment (gas phase) |
|----------|------------------|------------------------|
| Vertical IE, ΔSCF | 9.15 eV | ≈ 10.8–10.9 eV (vertical); 10.07 eV (adiabatic) |
| Vertical IE, Koopmans | 11.29 eV | (same) |
| Proton affinity | 222.3 kcal/mol | ≈ 204 kcal/mol |
| Deprotonation energy | +442.7 kcal/mol | ≈ 404 kcal/mol |

(Experimental values: NIST WebBook. The PA and deprotonation values are 298 K enthalpies.)

**Key insights:**
- **Ionization energy:** ΔSCF *underestimates* the IE because Hartree–Fock has no electron correlation, and the 10-electron neutral has more correlation energy than the 9-electron cation. Koopmans *overestimates* it because it ignores how the remaining electrons relax after one is removed. The two errors partly cancel in Koopmans' estimate.
- **Protonation** of NH₃ releases a large amount of energy (a big positive PA) — ammonia is a strong gas-phase base.
- **Deprotonation** costs about twice as much energy: you must separate two opposite charges (NH₂⁻ and H⁺).
- The comparison with experiment is rough: our numbers are electronic energies only (no zero-point vibrational energy or thermal corrections, which lower both PA and the deprotonation energy by several kcal/mol), at fixed geometries, with a modest basis set.
- The 6-31G deprotonation energy is off by almost 40 kcal/mol — mostly because **6-31G cannot describe the anion**. Let's fix that.

---

## Step 8: Anions Need Diffuse Functions

An anion's extra electron is held weakly, so its orbital is spread far from the nuclei. Standard basis sets like 6-31G are built for neutral atoms and have no functions that reach that far. **Diffuse functions** (the "+" in 6-31+G\*, "aug-" in aug-cc-pVDZ, "D" in def2-SVPD) fill that gap.

Diffuse basis sets QuantUI supports (from `config.SUPPORTED_BASIS_SETS`): **6-31+G\*, 6-31++G\*\*, aug-cc-pVDZ, def2-SVPD**.

A useful diagnostic: for a bound anion the **HOMO energy must be negative**. If it is zero or positive, the basis set is not really holding the extra electron.

We now repeat all five calculations with **6-31+G\*** (diffuse functions on N, plus d polarization functions).

### Run the cell below ⬇️ (a few seconds)

In [ ]:
print("Diffuse basis sets available:", [b for b in config.SUPPORTED_BASIS_SETS if config.is_diffuse_basis(b)])

basis_diffuse = "6-31+G*"
results_diffuse = {}
for name, mol in molecules.items():
    results_diffuse[name] = run_calc(mol, scf_method(mol), basis_diffuse)

rxn_diffuse = reaction_energies(results_diffuse)

print(f"\nCOMPARISON: {basis} vs {basis_diffuse}")
print("=" * 66)
print(f"{'Quantity':34s} {basis:>14s} {basis_diffuse:>14s}")
rows = [
    ("NH2- HOMO energy (eV)", homo_energy_ev(results["NH2-"]), homo_energy_ev(results_diffuse["NH2-"])),
    ("Deprotonation energy (kcal/mol)", rxn_631g["deprot_kcal"], rxn_diffuse["deprot_kcal"]),
    ("Proton affinity (kcal/mol)", rxn_631g["PA_kcal"], rxn_diffuse["PA_kcal"]),
    ("Vertical IE, ΔSCF (eV)", rxn_631g["IE_dSCF_eV"], rxn_diffuse["IE_dSCF_eV"]),
    ("Mulliken q(N) in NH3", results["NH3"].mulliken_charges[0], results_diffuse["NH3"].mulliken_charges[0]),
    ("Mulliken q(N) in NH2-", results["NH2-"].mulliken_charges[0], results_diffuse["NH2-"].mulliken_charges[0]),
]
for label, a, b in rows:
    print(f"{label:34s} {a:14.2f} {b:14.2f}")
print("=" * 66)
print("All converged:", all(r.converged for r in results_diffuse.values()))

### ✅ Checkpoint 5

**Expected:**

| Quantity | 6-31G | 6-31+G\* |
|----------|-------|----------|
| NH₂⁻ HOMO energy | ≈ 0.0 eV (+0.01) | −1.21 eV |
| Deprotonation energy | 442.7 kcal/mol | 421.2 kcal/mol |
| Proton affinity | 222.3 kcal/mol | 213.9 kcal/mol |
| Vertical IE (ΔSCF) | 9.15 eV | 9.40 eV |
| Mulliken q(N) in NH₃ | −0.91 | −1.16 |
| Mulliken q(N) in NH₂⁻ | −1.14 | −1.51 |

**Key insights:**
- With 6-31G the NH₂⁻ HOMO sits at about zero energy — the basis set is barely binding the extra electron. With diffuse functions the HOMO drops to −1.21 eV: a properly bound anion.
- The anion is stabilized much more than the neutral, so the deprotonation energy drops by about 21 kcal/mol, toward experiment (≈ 404 kcal/mol). The remaining gap is due to missing electron correlation, zero-point energy, and the fixed geometries.
- The cation-related quantities (PA, IE) change less — diffuse functions matter most for anions.
- **Mulliken charges change noticeably with the basis set.** Use them for trends within one basis, never as absolute numbers.

---

## Step 9: Electron Affinity — Diffuse Functions *and* Correlation

The **electron affinity** of the amino radical is the energy released when it captures an electron:

$$\text{NH}_2^\bullet + e^- \rightarrow \text{NH}_2^- \qquad EA = E(\text{NH}_2^\bullet) - E(\text{NH}_2^-)$$

A **positive EA** means the anion is more stable than radical + free electron. Experimentally EA(NH₂) = **0.771 eV**.

Hartree–Fock treats each electron in the *average* field of the others (no electron correlation). Adding an electron increases the correlation energy, so HF systematically favors the radical. **MP2** (second-order Møller–Plesset perturbation theory) adds correlation on top of HF. For the open-shell NH₂• radical, QuantUI's MP2 uses a restricted open-shell (ROHF) reference.

We compute the EA four ways: HF and MP2, each with 6-31G and 6-31+G\* (same NH₂ geometry for both species).

### Run the cell below ⬇️ (a few seconds)

In [ ]:
EA_EXPERIMENT_EV = 0.771

print("ELECTRON AFFINITY OF NH2•  (EA = E(NH2•) - E(NH2-))")
print("=" * 66)
print(f"{'Method':8s} {'Basis':9s} {'EA (eV)':>9s}  {'NH2• SCF':>10s}  Converged")
for method in ["HF", "MP2"]:
    for b in [basis, basis_diffuse]:
        radical_method = "UHF" if method == "HF" else "MP2"
        anion_method = "RHF" if method == "HF" else "MP2"
        r_rad = run_calc(mol_nh2_radical, radical_method, b)
        r_an = run_calc(mol_nh2, anion_method, b)
        ea = (r_rad.energy_hartree - r_an.energy_hartree) * HARTREE_TO_EV
        ok = r_rad.converged and r_an.converged
        print(f"{method:8s} {b:9s} {ea:+9.2f}  {r_rad.scf_variant:>10s}  {ok}")
print("=" * 66)
print(f"Experiment: EA = {EA_EXPERIMENT_EV} eV")

### ✅ Checkpoint 6

**Expected:**

| Method | 6-31G | 6-31+G\* |
|--------|-------|----------|
| HF  | −2.09 eV | −1.15 eV |
| MP2 | −1.32 eV | +0.32 eV |
| Experiment | | +0.771 eV |

**Key insights:**
- HF predicts a **negative** EA (NH₂⁻ unbound) in both basis sets — qualitatively wrong.
- Diffuse functions alone are not enough; correlation alone (MP2/6-31G) is not enough either.
- Only **MP2 with a diffuse basis** gives the correct sign. The remaining error comes from the modest basis set and the approximate treatment of correlation; larger diffuse basis sets (e.g. aug-cc-pVDZ) and higher-level methods move it closer to experiment.
- Electron affinities are a demanding test: you need **both** a diffuse basis and electron correlation.

---

## 🎓 Learning Review

### What You Accomplished

1. ✅ Built neutral, cationic, anionic, and radical species with the right charge and multiplicity
2. ✅ Ran every calculation directly in your own Python session
3. ✅ Learned why total energies of different species can't be ranked directly
4. ✅ Computed an ionization energy (ΔSCF and Koopmans), a proton affinity, and a deprotonation energy
5. ✅ Saw how diffuse functions and electron correlation change anion results

### Key Concepts

**Charge and Electrons:**
- Cation (+): fewer electrons; anion (−): more electrons
- Formula: $N_e = \sum Z_i - Q$
- Gaining/losing H⁺ keeps the electron count; gaining/losing e⁻ changes it

**Multiplicity Rules:**
- Even electron count → odd multiplicity (usually singlet)
- Odd electron count → even multiplicity (usually doublet) → use UHF
- Exceptions exist (e.g. triplet O₂ — Tutorial 3)

**Energy Comparisons:**
- Compare species only through balanced reactions
- E(H⁺) = 0 (no electrons), E(e⁻) = 0 (free electron at rest)
- Electronic energies still need zero-point and thermal corrections before comparing with experimental enthalpies

**Basis Sets for Anions:**
- Need diffuse functions: 6-31+G\*, 6-31++G\*\*, aug-cc-pVDZ, def2-SVPD (supported in QuantUI)
- Check: the anion's HOMO energy should be negative
- Electron affinities also need electron correlation (e.g. MP2)

### Practical Tips

**Common Charged Species:**
- **Acids:** Compare HA vs A⁻ (use a diffuse basis for A⁻!)
- **Bases:** Compare B vs BH⁺
- **Radical ions:** Formed by removing/adding one electron → doublets, UHF
- **Metal ions:** Many oxidation states — check the multiplicity carefully

---

## 🚀 Next Experiments

### Try These Variations

1. **Water protonation series:**
   - OH⁻, H₂O, H₃O⁺ — another 10-electron isoelectronic series
   - Compute the proton affinity of water and the deprotonation energy, with 6-31G and 6-31+G\*

2. **Bigger diffuse basis:**
   - Repeat the NH₂ electron affinity with MP2/aug-cc-pVDZ
   - Does it move closer to 0.771 eV?

3. **Carboxylate anion:**
   - HCOOH (formic acid) vs HCOO⁻ (formate)
   - Compute the gas-phase deprotonation energy and compare with NH₃ — which is the stronger gas-phase acid?

4. **Metal ions:**
   - Na (doublet) and Na⁺ (singlet)
   - Calculate the ionization energy with ΔSCF and compare to experiment (5.14 eV)

5. **Use the main app:** open [molecule_computations.ipynb](../molecule_computations.ipynb), set the charge and multiplicity in the molecule panel, and run the same species interactively.

---

## 💬 Discussion Questions

1. **pKa Prediction:** How would you estimate the pKa of NH₄⁺ (or NH₃) from these calculations? What additional information do you need? (Hint: solvation free energies of every species, including H⁺.)

2. **Solvation:** These are gas-phase calculations. How would water change the relative energies of NH₃, NH₄⁺, and NH₂⁻? Which species would be stabilized most, and why?

3. **Anion Stability:** Why do anions need diffuse basis functions? Why was the NH₂⁻ HOMO energy a useful warning sign in 6-31G?

4. **Koopmans vs. ΔSCF:** One method ignores orbital relaxation, the other ignores electron correlation. Which error makes each IE too high or too low?

5. **Isoelectronic Series:** Compare the 10-electron series Ne, F⁻, Na⁺. Which holds its electrons most tightly? How does that relate to the HOMO-LUMO gaps of NH₂⁻, NH₃, NH₄⁺?

6. **Biological Relevance:** Amino acid side chains have different pKa values. How do calculations like these help us understand protonation states at biological pH?

---

## 📖 Further Reading

- C. J. Cramer, *Essentials of Computational Chemistry*, 2nd ed. (Wiley, 2004) — basis sets (diffuse functions), Koopmans' theorem, and ΔSCF.
- A. Szabo and N. S. Ostlund, *Modern Quantum Chemistry* (Dover, 1996) — Koopmans' theorem and MP2.
- NIST Chemistry WebBook (https://webbook.nist.gov) — experimental ionization energies, proton affinities, gas-phase acidities, and electron affinities.
- NIST CCCBDB (https://cccbdb.nist.gov) — computed vs. experimental data for many small molecules and ions.

---

## ✅ Tutorial Complete!

**Well done!** You can now handle charged species in quantum chemistry calculations.

This opens up applications in:
- Acid-base chemistry
- Electrochemistry
- Mass spectrometry
- Biochemistry (protonation states)
- Ion atmospheres and catalysis

**Ready for the finale?** → [Tutorial 5: Comparing Results](05_comparing_results.ipynb)

**Want to explore on your own?** → [Main QuantUI app](../molecule_computations.ipynb)

---

*Questions? Contact your instructor or create a GitHub issue.*